# Stabilizer Python: Ultra-Detailed How-To Notebook

This notebook is a practical guide to the local `stabilizer-python/` package in this repository. It is written as a hands-on lab: each section explains the stabilizer concept, then uses the actual package API to inspect tableaus, phase bits, Clifford gates, measurements, and small error-correction workflows.

The package is intentionally small and transparent. Its core object is `StabilizerState`, an Aaronson-Gottesman style stabilizer tableau with:

- `x_mat`: the binary X part of each Pauli row.
- `z_mat`: the binary Z part of each Pauli row.
- `r_phase`: one sign bit per row, where `0` means `+` and `1` means `-`.
- `2n` rows for `n` qubits: the first `n` rows are destabilizers, and the last `n` rows are stabilizers.

By the end, you should be able to create states, apply Clifford gates, read raw matrices, reason about phase updates, measure qubits, build circuits, and walk through the included error-correction helpers.

## 0. Big Picture

A stabilizer simulator does not store a full length-`2**n` statevector. Instead, it stores a compact generating set for the Pauli operators that fix the quantum state.

For example, `|0>` is stabilized by `+Z`, because `Z|0> = |0>`. The state `|+>` is stabilized by `+X`, because `X|+> = |+>`. A Bell state can be described by two commuting stabilizers, such as `+XX` and `+ZZ`.

This package stores those generators in binary form. Each row is a Pauli string:

- `x=0, z=0` means `I`.
- `x=1, z=0` means `X`.
- `x=0, z=1` means `Z`.
- `x=1, z=1` means `Y`, with the row sign held separately in `r_phase`.

The package tracks only real row signs, `+` and `-`, because stabilizer-state tableaus do not need arbitrary complex amplitudes.

In [ ]:
# Notebook setup: find the repository root and import the local package.
from pathlib import Path
import random
import sys
from typing import Iterable, List, Tuple


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "stabilizer-python" / "stabilizer_python").exists():
            return candidate
    raise RuntimeError("Could not find repository root containing stabilizer-python/")


REPO_ROOT = find_repo_root()
PKG_ROOT = REPO_ROOT / "stabilizer-python"
if str(PKG_ROOT) not in sys.path:
    sys.path.insert(0, str(PKG_ROOT))

from stabilizer_python import Circuit, StabilizerState, gaussian_elimination_gf2, rank_gf2
from stabilizer_python.codes import BitFlip3Code, Shor9Code, run_2qubit_bell

print("Repository root:", REPO_ROOT)
print("Package root:", PKG_ROOT)

## 1. Debug Helpers Used Throughout

The package already provides useful formatters:

- `format_chp_printstate()` prints signed Pauli rows, similar to CHP output.
- `format_xz_binary_matrices()` prints the raw X and Z matrices.
- `format_phase_matrix()` prints the phase column.
- `format_tableau_debug()` prints all of the above together.

The helper functions below add a few notebook-friendly summaries and validation checks.

In [ ]:
def pauli_char(x_bit: int, z_bit: int) -> str:
    if x_bit == 0 and z_bit == 0:
        return "I"
    if x_bit == 1 and z_bit == 0:
        return "X"
    if x_bit == 0 and z_bit == 1:
        return "Z"
    return "Y"


def pauli_string(x_row: List[int], z_row: List[int]) -> str:
    return "".join(pauli_char(x, z) for x, z in zip(x_row, z_row))


def stabilizer_strings(state: StabilizerState) -> List[str]:
    rows = []
    for phase, x_row, z_row in state.stabilizer_generators():
        sign = "-" if phase else "+"
        rows.append(sign + pauli_string(x_row, z_row))
    return rows


def show_state(state: StabilizerState, title: str = "Tableau") -> None:
    print("=" * len(title))
    print(title)
    print("=" * len(title))
    print(state.format_tableau_debug())
    print()
    print("Stabilizer generators:", stabilizer_strings(state))


def symplectic_product(state: StabilizerState, row_a: int, row_b: int) -> int:
    total = 0
    for q in range(state.n):
        total ^= state.x_mat[row_a][q] & state.z_mat[row_b][q]
        total ^= state.z_mat[row_a][q] & state.x_mat[row_b][q]
    return total


def assert_valid_stabilizer_tableau(state: StabilizerState) -> None:
    assert len(state.x_mat) == 2 * state.n
    assert len(state.z_mat) == 2 * state.n
    assert len(state.r_phase) == 2 * state.n

    for row in state.x_mat + state.z_mat:
        assert len(row) == state.n
        assert all(bit in (0, 1) for bit in row)
    assert all(phase in (0, 1) for phase in state.r_phase)

    stabilizer_matrix = []
    for row_a in range(state.n, 2 * state.n):
        row = state.x_mat[row_a] + state.z_mat[row_a]
        stabilizer_matrix.append(row)
        assert any(row), "stabilizer generator cannot be identity"
        for row_b in range(row_a + 1, 2 * state.n):
            assert symplectic_product(state, row_a, row_b) == 0, "stabilizers must commute"

    assert rank_gf2(stabilizer_matrix) == state.n, "stabilizer rows must be independent"

## 2. Creating the Zero State

`StabilizerState.zero(n)` creates `|0...0>`.

For `n` qubits:

- Destabilizer rows are initialized as `X_i`.
- Stabilizer rows are initialized as `Z_i`.
- All phase bits start at `0`, so all row signs are positive.

This is the canonical starting point for nearly every example in this package.

In [ ]:
st = StabilizerState.zero(3)
show_state(st, "Initial |000> tableau")
assert_valid_stabilizer_tableau(st)

## 3. Reading the X, Z, and Phase Matrices

A row is interpreted column-by-column. For a 3-qubit state, a row with:

- X bits `[1, 0, 1]`
- Z bits `[0, 1, 1]`

means `XZY`, because the third qubit has both X and Z bits set, which represents `Y`.

The phase column tells whether the whole Pauli row is positive or negative. A phase bit of `1` means the row has a leading minus sign.

In [ ]:
print(st.format_xz_binary_matrices())
print()
print(st.format_phase_matrix())
print()
print("CHP-style signed rows:")
print(st.format_chp_printstate())

## 4. Single-Qubit Clifford Gates

The tableau implements several single-qubit Clifford gates directly:

- `i(q)`: identity.
- `x(q)`, `y(q)`, `z(q)`: Pauli gates.
- `h(q)`: Hadamard.
- `s(q)`: phase gate.
- `sdg(q)` / `s_dagger(q)`: inverse phase gate.
- `sx(q)` / `sqrt_x(q)`: square-root of X.
- `sxdg(q)` / `sqrt_x_dagger(q)`: inverse square-root of X.

These gates update the tableau by conjugating every Pauli row. For example, `H` swaps X and Z bits on the target qubit, and flips the row sign for Y because `HYH = -Y`.

In [ ]:
single = StabilizerState.zero(1)
show_state(single, "One-qubit |0>")

single.h(0)
show_state(single, "After H: |+> stabilized by +X")

single.s(0)
show_state(single, "After S: +X maps to +Y")

single.sdg(0)
show_state(single, "After Sdg: back to +X")

single.sx(0)
show_state(single, "After sqrt-X")

## 5. Two-Qubit Clifford Gates

The tableau supports these two-qubit Clifford operations:

- `cnot(control, target)` and alias `cx(control, target)`.
- `cz(control, target)`.
- `cy(control, target)`.
- `swap(q1, q2)`.

Some are implemented directly, and some are decomposed into trusted Clifford primitives. For example, `CZ` is `H(target) -> CNOT(control, target) -> H(target)`.

In [ ]:
two = StabilizerState.zero(2)
two.h(0)
two.cnot(0, 1)
show_state(two, "Bell state from H(0), CNOT(0, 1)")

cz_demo = StabilizerState.zero(2)
cz_demo.h(0)
cz_demo.h(1)
cz_demo.cz(0, 1)
show_state(cz_demo, "CZ acting on |++>")

swap_demo = StabilizerState.zero(2)
swap_demo.x(0)
print("Before SWAP stabilizers:", stabilizer_strings(swap_demo))
swap_demo.swap(0, 1)
print("After SWAP stabilizers:", stabilizer_strings(swap_demo))

## 6. Phase Bits Are Where Syndrome Information Lives

For stabilizer code examples, phase bits often encode the syndrome. If an error anticommutes with a stabilizer generator, that generator's sign flips from `+` to `-`.

In this package:

- `r_phase[row] == 0` means `+`.
- `r_phase[row] == 1` means `-`.
- The stabilizer rows are `state.n` through `2 * state.n - 1`.

The next cell shows a simple sign flip caused by applying `X` to `|0>`. Since `XZX = -Z`, the stabilizer `+Z` becomes `-Z`, which corresponds to `|1>`.

In [ ]:
one = StabilizerState.zero(1)
show_state(one, "Before X: |0>")

one.x(0)
show_state(one, "After X: |1>, stabilized by -Z")

## 7. Building Circuits with `Circuit`

`Circuit` is a small chained-operation wrapper. It stores operations and applies them to a `StabilizerState` with `run(state)`.

Currently the wrapper supports:

- `h(q)`
- `s(q)`
- `x(q)`
- `z(q)`
- `cnot(control, target)`
- `mz(q, key=None)` for Z-basis measurement

The low-level `StabilizerState` has more gates than `Circuit`; when you need the newer gates like `y`, `sdg`, `sx`, `cz`, `cy`, or `swap`, call them directly on the state unless the wrapper has been extended.

In [ ]:
circuit_state = StabilizerState.zero(3)
circuit = Circuit(3).h(0).cnot(0, 1).cnot(0, 2)
measurements = circuit.run(circuit_state)

show_state(circuit_state, "GHZ-style state from Circuit")
print("Measurements returned by this circuit:", measurements)

## 8. Measuring Z on One Qubit

`measure_z(q)` measures qubit `q` in the computational basis and returns a bit:

- `0` means the `+1` eigenvalue of Z.
- `1` means the `-1` eigenvalue of Z.

If the outcome is deterministic, the method computes it from the stabilizer tableau. If it is random, the method samples `0` or `1`, then updates the tableau to reflect the post-measurement state.

For reproducible notebook output, seed Python's `random` module before examples with genuinely random outcomes.

In [ ]:
random.seed(7)

m_state = StabilizerState.zero(1)
print("Measuring |0> in Z is deterministic:", m_state.measure_z(0))
show_state(m_state, "After deterministic Z measurement")

random_state = StabilizerState.zero(1)
random_state.h(0)
print("Measuring |+> in Z is random with this seed:", random_state.measure_z(0))
show_state(random_state, "After random Z measurement of |+>")

## 9. Measuring Parity with an Ancilla

A common stabilizer-code pattern is to measure a multi-qubit Pauli parity using an ancilla.

For a Z-parity check `Z0 Z1`:

1. Prepare an ancilla in `|0>`.
2. CNOT from data qubit 0 into the ancilla.
3. CNOT from data qubit 1 into the ancilla.
4. Measure Z on the ancilla.

The measured bit tells the parity:

- `0` means even / `+1` eigenvalue.
- `1` means odd / `-1` eigenvalue.

In [ ]:
parity_state = StabilizerState.zero(3)
parity_state.x(1)  # prepare |010>, so Z0 Z1 parity should be odd

Circuit(3).cnot(0, 2).cnot(1, 2).run(parity_state)
parity = parity_state.measure_z(2)

print("Measured Z0 Z1 parity bit:", parity)
show_state(parity_state, "After measuring Z0Z1 using ancilla q2")

## 10. Bell State Walkthrough

The package includes `run_2qubit_bell()`, which prepares the Bell state `|Phi+>` using:

1. `H(0)`
2. `CNOT(0, 1)`

The expected stabilizer generators are `+XX` and `+ZZ`.

In [ ]:
bell, _ = run_2qubit_bell()
show_state(bell, "Bell state |Phi+>")
assert set(stabilizer_strings(bell)) == {"+XX", "+ZZ"}

## 11. GHZ State Walkthrough

A 3-qubit GHZ state is created by applying `H` to one root qubit and CNOTing that root into the others.

The expected stabilizers are generated by strings equivalent to:

- `+XXX`
- `+ZZI`
- `+ZIZ`

The exact row order can depend on the circuit construction, so compare sets rather than relying on row order.

In [ ]:
ghz = StabilizerState.zero(3)
Circuit(3).h(0).cnot(0, 1).cnot(0, 2).run(ghz)
show_state(ghz, "3-qubit GHZ state")
assert set(stabilizer_strings(ghz)) == {"+XXX", "+ZZI", "+ZIZ"}

## 12. The 3-Qubit Bit-Flip Code

`BitFlip3Code` models the repetition code that protects against one X error.

Logical states:

- `|0_L> = |000>`
- `|1_L> = |111>`

Stabilizer checks:

- `Z0 Z1`
- `Z1 Z2`

Syndrome interpretation:

- `(0, 0)`: no detected X error.
- `(1, 0)`: X error on qubit 0.
- `(1, 1)`: X error on qubit 1.
- `(0, 1)`: X error on qubit 2.

In [ ]:
encoded = StabilizerState.zero(5)  # data q0..q2, ancillas q3..q4
BitFlip3Code.encoder_circuit().run(encoded)
show_state(encoded, "3-qubit bit-flip code with ancillas")

encoded.x(1)
show_state(encoded, "After injected X error on data qubit 1")

s01, s12 = BitFlip3Code.measure_syndrome(encoded)
print("Measured syndrome:", (s01, s12))

BitFlip3Code.correct_x_from_syndrome(encoded, s01, s12)
print("Syndrome after correction:", BitFlip3Code.measure_syndrome(encoded))
show_state(encoded, "After correction")

## 13. Direct Syndrome Reading Without Extra Ancillas

The helper `BitFlip3Code.read_syndrome(state)` reads syndrome information directly from stabilizer generator phase bits on a 3-qubit data state.

This is not the same physical procedure as an ancilla measurement circuit, but it is useful for debugging and for seeing why phase bits matter.

In [ ]:
direct = StabilizerState.zero(3)
BitFlip3Code.encoder_circuit().run(direct)
print("No-error direct syndrome:", BitFlip3Code.read_syndrome(direct))

direct.x(2)
show_state(direct, "3-qubit code after X on q2")
print("Direct syndrome after X on q2:", BitFlip3Code.read_syndrome(direct))

## 14. Why Z Errors Are Not Detected by This Code

The bit-flip repetition code checks Z-parities to detect X errors. A Z error commutes with those Z-parity checks, so the syndrome remains trivial.

This is a feature of the code being demonstrated, not a simulator bug. To detect phase flips, you need checks in the X basis or a larger code such as Shor's code.

In [ ]:
for q in range(3):
    phase_error_state = StabilizerState.zero(5)
    BitFlip3Code.encoder_circuit().run(phase_error_state)
    phase_error_state.z(q)
    syndrome = BitFlip3Code.measure_syndrome(phase_error_state)
    print(f"Z error on q{q}: syndrome {syndrome}")

## 15. Shor 9-Qubit Code Overview

`Shor9Code.encoder_circuit()` creates a 9-qubit Shor-code encoding circuit. The implementation uses this layout:

- Phase-protection roots: `q0`, `q3`, `q6`.
- Bit-flip repetition blocks: `(q0, q1, q2)`, `(q3, q4, q5)`, `(q6, q7, q8)`.

The current helper focuses on reading and correcting single X-error syndromes by matching stabilizer phase-bit patterns.

In [ ]:
shor = StabilizerState.zero(9)
Shor9Code.encoder_circuit().run(shor)
print("Initial Shor syndrome:", Shor9Code.read_syndrome(shor))

shor.x(4)
print("Syndrome after X on q4:", Shor9Code.read_syndrome(shor))

Shor9Code.correct_x_from_syndrome(shor, Shor9Code.read_syndrome(shor))
print("Syndrome after correction:", Shor9Code.read_syndrome(shor))
assert_valid_stabilizer_tableau(shor)

## 16. Validating Random Clifford Circuits

A good stabilizer-tableau sanity check is:

1. Generate a random Clifford circuit.
2. Apply it to `|0...0>`.
3. Verify the stabilizer rows commute.
4. Verify the stabilizer generator matrix has full rank.

The helper below mirrors the repository tests, but it is written as notebook code so you can modify depth, qubit count, or gate set interactively.

In [ ]:
def apply_gate(state: StabilizerState, gate: Tuple[str, Tuple[int, ...]]) -> None:
    name, targets = gate
    if name == "H":
        state.h(targets[0])
    elif name == "S":
        state.s(targets[0])
    elif name == "X":
        state.x(targets[0])
    elif name == "Y":
        state.y(targets[0])
    elif name == "Z":
        state.z(targets[0])
    elif name == "SDG":
        state.sdg(targets[0])
    elif name == "SX":
        state.sx(targets[0])
    elif name == "SXDG":
        state.sxdg(targets[0])
    elif name == "CNOT":
        state.cnot(targets[0], targets[1])
    elif name == "CZ":
        state.cz(targets[0], targets[1])
    elif name == "CY":
        state.cy(targets[0], targets[1])
    elif name == "SWAP":
        state.swap(targets[0], targets[1])
    else:
        raise ValueError(f"unknown gate {name}")


def random_clifford_gates(rng: random.Random, n_qubits: int, depth: int) -> List[Tuple[str, Tuple[int, ...]]]:
    one_qubit = ["H", "S", "X", "Y", "Z", "SDG", "SX", "SXDG"]
    two_qubit = ["CNOT", "CZ", "CY", "SWAP"]
    gates = []
    for _ in range(depth):
        if n_qubits >= 2 and rng.random() < 0.35:
            name = rng.choice(two_qubit)
            q0, q1 = rng.sample(range(n_qubits), 2)
            gates.append((name, (q0, q1)))
        else:
            name = rng.choice(one_qubit)
            gates.append((name, (rng.randrange(n_qubits),)))
    return gates


rng = random.Random(123)
rand_state = StabilizerState.zero(4)
rand_gates = random_clifford_gates(rng, n_qubits=4, depth=25)
for gate in rand_gates:
    apply_gate(rand_state, gate)

print("Random gates:")
for gate in rand_gates:
    print(" ", gate)
print()
show_state(rand_state, "Random Clifford tableau")
assert_valid_stabilizer_tableau(rand_state)

## 17. Gaussian Elimination over GF(2)

The package includes two small linear-algebra utilities:

- `gaussian_elimination_gf2(matrix)` returns a reduced row-echelon form and pivot columns.
- `rank_gf2(matrix)` returns the binary rank.

These are useful when checking whether stabilizer generators are independent.

In [ ]:
matrix = [
    [1, 0, 1, 1],
    [0, 1, 1, 0],
    [1, 1, 0, 1],
]

rref, pivots = gaussian_elimination_gf2(matrix)
print("Input matrix:")
for row in matrix:
    print(row)
print("\nRREF over GF(2):")
for row in rref:
    print(row)
print("Pivot columns:", pivots)
print("Rank:", rank_gf2(matrix))

## 18. Gate Decompositions to Remember

Some useful Clifford equivalences used by the package and tests:

- `Sdg = S S S`
- `SX = H S H`
- `SXdg = H Sdg H`
- `Y` has the same conjugation action as `X` followed by `Z`, ignoring global phase.
- `CX = CNOT`
- `CZ = H(target) CNOT(control, target) H(target)`
- `CY = Sdg(target) CNOT(control, target) S(target)`
- `SWAP = CNOT(q1, q2) CNOT(q2, q1) CNOT(q1, q2)`

Because stabilizer simulation ignores global phase, decompositions that differ only by a global phase are equivalent for tableau evolution.

In [ ]:
def same_tableau(left: StabilizerState, right: StabilizerState) -> bool:
    return (
        left.n == right.n
        and left.x_mat == right.x_mat
        and left.z_mat == right.z_mat
        and left.r_phase == right.r_phase
    )

base = StabilizerState.zero(3)
base.h(0)
base.s(0)
base.h(1)
base.cnot(0, 1)
base.s(2)

left = base.copy()
right = base.copy()
left.cz(0, 2)
right.h(2)
right.cnot(0, 2)
right.h(2)
print("CZ equals H-CNOT-H decomposition:", same_tableau(left, right))

left = base.copy()
right = base.copy()
left.sx(1)
right.h(1)
right.s(1)
right.h(1)
print("SX equals H-S-H decomposition:", same_tableau(left, right))

## 19. Common Pitfalls

### Row order is not physics

Two equivalent stabilizer states can sometimes be represented by different generator choices. Many tests in this repository compare exact tableaus because the code paths are deterministic, but for conceptual comparisons it is often better to compare generated stabilizer groups or known observables.

### Phase bits are syndrome bits

Do not ignore `r_phase`. A sign flip from `+ZZI` to `-ZZI` is exactly the information used to locate many errors.

### `Circuit` is smaller than `StabilizerState`

The low-level tableau has more Clifford methods than the current `Circuit` wrapper. If you add a new gate to the tableau and want chainable circuits, extend `Circuit` too.

### Stabilizer simulation is Clifford-only

This package is for Clifford gates, Pauli measurements, and stabilizer states. Non-Clifford gates such as `T` cannot be represented exactly with this tableau-only simulator.

### Measurement changes the state

`measure_z(q)` is not a read-only query. It returns an outcome and updates the tableau to the corresponding post-measurement state.

## 20. Suggested Exercises

Try these modifications after running the notebook once:

1. Extend `Circuit` with `y`, `sdg`, `sx`, `cz`, `cy`, and `swap`, then update the random circuit helper to use `Circuit` instead of direct state calls.
2. Add a helper that computes all stabilizers generated by the `n` stabilizer rows for small `n`, then compare states up to generator choice.
3. Build an X-basis parity measurement by surrounding data qubits with `H` gates.
4. Create a notebook cell that injects every single-qubit X error into the Shor code and prints the syndrome table.
5. Add visual plotting of the X, Z, and phase matrices with `matplotlib.imshow`.
6. Write a regression test that executes selected notebook examples as Python snippets.

## 21. Quick Reference

### Core imports

```python
from stabilizer_python import StabilizerState, Circuit
from stabilizer_python.codes import BitFlip3Code, Shor9Code, run_2qubit_bell
```

### Core state methods

```python
st = StabilizerState.zero(n)
st.copy()
st.h(q); st.s(q); st.sdg(q); st.sx(q); st.sxdg(q)
st.x(q); st.y(q); st.z(q)
st.cnot(c, t); st.cx(c, t); st.cy(c, t); st.cz(c, t); st.swap(a, b)
st.measure_z(q)
st.reset_z(q)
st.stabilizer_generators()
st.format_tableau_debug()
```

### Circuit methods

```python
Circuit(n).h(q).s(q).x(q).z(q).cnot(c, t).mz(q).run(state)
```

### Matrix interpretation

- X and Z matrices together encode the Pauli letters.
- Phase matrix encodes row signs.
- Stabilizer rows are the bottom half of the tableau.
- Destabilizer rows are the top half.